# Credit Approval — Evaluation, Explainability & Fairness
Final evaluation on hold-out set, SHAP analysis, calibration, and fairness assessment.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.utils import load_raw_data, TARGET_COL, get_optimal_threshold
from src.preprocessing import build_preprocessing_pipeline, get_feature_names
from src.evaluation import (
    compute_all_metrics, plot_roc_curves, plot_pr_curves, plot_calibration,
    plot_ks_chart, plot_confusion_matrix, compute_shap_values,
    plot_shap_summary, plot_shap_dependence, fairness_analysis, build_results_table
)
from sklearn.model_selection import train_test_split

df = load_raw_data("../data/raw/crx.data")
X_raw = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].values

X_tr_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.15, stratify=y, random_state=42
)

preproc = joblib.load("../outputs/models/preprocessor.pkl")
X_train = preproc.transform(X_tr_raw)
X_test  = preproc.transform(X_test_raw)
feature_names = get_feature_names(preproc)

ensemble = joblib.load("../outputs/models/stacking_ensemble.pkl")
lgb_model = joblib.load("../outputs/models/lightgbm_best.pkl")
print("Models loaded.")

In [ ]:
# Hold-out evaluation
test_scores = ensemble.predict_proba(X_test)[:, 1]
threshold = get_optimal_threshold(y_test, test_scores)
y_pred = (test_scores >= threshold).astype(int)

metrics = compute_all_metrics(y_test, test_scores, threshold)
print("Hold-out metrics:")
for k, v in metrics.items():
    print(f"  {k:15s}: {v}")

In [ ]:
# ROC and PR curves (using test scores for final evaluation)
plot_roc_curves({"StackingEnsemble": test_scores}, y_test, "../outputs/reports/roc_curves_final.png")
plot_pr_curves({"StackingEnsemble": test_scores}, y_test, "../outputs/reports/pr_curves_final.png")
plot_calibration({"StackingEnsemble": test_scores}, y_test, "../outputs/reports/calibration_final.png")
print("Comparison plots saved.")

In [ ]:
# KS chart + confusion matrix
plot_ks_chart(y_test, test_scores, "StackingEnsemble", "../outputs/reports/ks_chart.png")
plot_confusion_matrix(y_test, y_pred, "StackingEnsemble", "../outputs/reports/confusion_matrix.png")
print("KS chart and confusion matrix saved.")

In [ ]:
# SHAP analysis (LightGBM)
print("Computing SHAP values...")
shap_vals, explainer, X_sample = compute_shap_values(lgb_model, X_test, feature_names)
plot_shap_summary(shap_vals, X_sample, feature_names, "../outputs/reports/shap_summary.png")
plot_shap_dependence(shap_vals, X_sample, feature_names, top_n=3,
                     save_path_prefix="../outputs/reports/shap_dep")
print("SHAP plots saved.")

In [ ]:
# Fairness analysis
fairness_df = fairness_analysis(
    X_test_raw.reset_index(drop=True), y_test, test_scores,
    group_col="A1",
    threshold=threshold,
    save_path="../outputs/reports/fairness_analysis.csv"
)
fairness_df

## Fairness Interpretation

The Disparate Impact Ratio measures whether one demographic group is approved at less than 80% the rate of the most-approved group (the "80% rule" from EEOC guidelines).

- **DIR ≥ 0.8**: Acceptable under current regulatory guidance
- **DIR < 0.8**: Potential adverse impact — warrants investigation

Note: The UCI dataset anonymizes sensitive attributes; A1 is a binary attribute (likely gender proxy). Results here demonstrate the fairness evaluation methodology. In production, a formal bias audit with legal guidance would be required.

In [ ]:
# Threshold comparison: F1-optimal vs. fairness-optimal
from src.utils import get_optimal_threshold
import numpy as np

print(f"F1-optimal threshold: {threshold:.3f}")

# Find threshold that minimizes approval rate disparity between A1 groups
group = X_test_raw.reset_index(drop=True)["A1"].values
thresholds_to_try = np.linspace(0.2, 0.8, 61)
disparities = []
for t in thresholds_to_try:
    yp = (test_scores >= t).astype(int)
    rates = {}
    for g in ["a", "b"]:
        mask = group == g
        if mask.sum() > 0:
            rates[g] = yp[mask].mean()
    if len(rates) == 2:
        disparities.append(abs(rates["a"] - rates["b"]))
    else:
        disparities.append(1.0)

best_fair_t = thresholds_to_try[np.argmin(disparities)]
print(f"Fairness-optimal threshold: {best_fair_t:.3f}")
print(f"Min approval rate disparity: {min(disparities):.4f}")

## Summary

| Metric | Value | Target | Status |
|--------|-------|--------|---------|
| AUC-ROC | (see above) | ≥ 0.87 | |
| Gini | (see above) | ≥ 0.74 | |
| KS | (see above) | ≥ 0.50 | |
| ECE | (see above) | < 0.05 | |

The stacking ensemble of LightGBM + XGBoost + CatBoost with a Logistic Regression meta-learner provides the best discrimination while maintaining reasonable calibration.